In [3]:
import torch
import os
from torch import nn
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm
import numpy as np




class Tetris_shapes(nn.Module):
    def __init__(self,input_features, output_features, hidden_units):
        super().__init__()
        self.linear_layer_stack = nn.Sequential(nn.Linear(in_features=input_features, out_features=hidden_units),
                                                nn.ReLU(),
                                                nn.Linear(in_features=hidden_units, out_features=hidden_units),
                                                nn.ReLU(),
                                                nn.Linear(in_features=hidden_units, out_features=output_features))

        for layer in self.linear_layer_stack:
          if isinstance(layer, nn.Linear):
            nn.init.kaiming_uniform_(layer.weight)



    def forward(self, x):
        return self.linear_layer_stack(x)


model_linear_correlated_data = Tetris_shapes(input_features=64, output_features=2, hidden_units=10)

model_linear_correlated_data_dict = torch.load("/content/linear_1d1p_0.0125_correlated.pt")


def collect_activation(model,x):
  x= torch.tensor(np.array(x, dtype=np.float32))
  with torch.no_grad():
    layers = [model.linear_layer_stack[i] for i in range(len(model.linear_layer_stack))]
    activation = []
    for layer in layers:
      x = layer(x)
      activation.append(x)
  return activation

#This function trains a given model in batches and print out the results at every 10th epoch
def training_model_dicts(model: torch.nn.Module, model_dict: dict, epochs : int = 10):

    x_train = torch.tensor(np.array(model_dict["x_train"], dtype=np.float32))
    y_train = torch.tensor(np.array(model_dict["y_train"], dtype=np.int64))
    x_test = torch.tensor(np.array(model_dict["x_val"], dtype=np.float32))
    y_test = torch.tensor(np.array(model_dict["y_val"], dtype=np.int64))


    train_dataset = TensorDataset(x_train, y_train)
    test_dataset = TensorDataset(x_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    results = {"train_loss": [],
             "train_acc": [],
             "test_loss": [],
             "test_acc": []
             }


    for epochs in tqdm(range(epochs)):
        model.train()
        train_loss = 0
        train_acc = 0
        for batch ,(x_batch, y_batch) in enumerate(train_loader):
            y_pred = model(x_batch)
            loss = loss_fn(y_pred, y_batch)
            train_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            train_acc += (y_pred_class == y_batch).sum().item() / len(y_pred)

        train_loss /= len(train_loader)
        train_acc /= len(train_loader)

        model.eval()
        test_loss = 0
        test_acc = 0
        with torch.inference_mode():
            total_loss = 0
            for batch ,(x_batch, y_batch) in enumerate(test_loader):
                test_y_pred = model(x_batch)
                loss = loss_fn(test_y_pred, y_batch)
                test_loss += loss.item()
                y_pred_class = torch.argmax(torch.softmax(test_y_pred, dim=1), dim=1)
                test_acc += (y_pred_class == y_batch).sum().item() / len(test_y_pred)

            test_loss /= len(test_loader)
            test_acc /= len(test_loader)

        if(epochs % 10 == 0):
          print(
          f"Epoch: {epochs} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
          )


        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    return results




def save_acts_and_model(acts,model,model_path,act_path):
  torch.save(acts,act_path)
  torch.save(model.state_dict(),model_path)

<ipython-input-3-82ce14b1c6aa>:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_linear_correlated_data_dict = torch.load("/content/linear_1d1p_0.0125_correlated.pt")


In [4]:
training_model_dicts(model=model_linear_correlated_data, model_dict=model_linear_correlated_data_dict, epochs=100)
acts = collect_activation(model=model_linear_correlated_data,x=model_linear_correlated_data_dict["x_train"])

save_acts_and_model(acts=acts,model=model_linear_correlated_data,model_path="linear_correlated_model.pt",act_path="acts_linear_correlated_model.pt")


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch: 0 | train_loss: 0.6953 | train_acc: 0.5131 | test_loss: 0.6916 | test_acc: 0.5215
Epoch: 10 | train_loss: 0.3227 | train_acc: 0.9768 | test_loss: 0.2432 | test_acc: 1.0000
Epoch: 20 | train_loss: 0.0071 | train_acc: 1.0000 | test_loss: 0.0068 | test_acc: 1.0000
Epoch: 30 | train_loss: 0.0010 | train_acc: 1.0000 | test_loss: 0.0009 | test_acc: 1.0000
Epoch: 40 | train_loss: 0.0002 | train_acc: 1.0000 | test_loss: 0.0002 | test_acc: 1.0000
Epoch: 50 | train_loss: 0.0482 | train_acc: 0.9905 | test_loss: 0.3551 | test_acc: 0.9043
Epoch: 60 | train_loss: 0.0001 | train_acc: 1.0000 | test_loss: 0.0001 | test_acc: 1.0000
Epoch: 70 | train_loss: 0.0000 | train_acc: 1.0000 | test_loss: 0.0000 | test_acc: 1.0000
Epoch: 80 | train_loss: 0.0000 | train_acc: 1.0000 | test_loss: 0.0000 | test_acc: 1.0000
Epoch: 90 | train_loss: 0.0000 | train_acc: 1.0000 | test_loss: 0.0000 | test_acc: 1.0000


In [14]:
import torch
random_acts = []
for act in acts:
  random_acts.append(torch.rand_like(act))
random_acts

[tensor([[0.2141, 0.2257, 0.3735,  ..., 0.0925, 0.3016, 0.5205],
         [0.6349, 0.3697, 0.3344,  ..., 0.2730, 0.9315, 0.0372],
         [0.5495, 0.5156, 0.3084,  ..., 0.7112, 0.0739, 0.9182],
         ...,
         [0.1482, 0.7793, 0.2747,  ..., 0.1348, 0.3662, 0.8802],
         [0.2942, 0.0226, 0.5145,  ..., 0.2753, 0.4686, 0.0795],
         [0.2673, 0.3139, 0.6458,  ..., 0.7109, 0.8728, 0.6454]]),
 tensor([[0.0941, 0.1949, 0.4542,  ..., 0.8862, 0.8207, 0.5029],
         [0.0677, 0.8977, 0.8472,  ..., 0.5791, 0.0389, 0.5426],
         [0.1652, 0.9496, 0.0729,  ..., 0.6626, 0.7680, 0.5560],
         ...,
         [0.5142, 0.3886, 0.5643,  ..., 0.2166, 0.7622, 0.0663],
         [0.2302, 0.9634, 0.6826,  ..., 0.5617, 0.2084, 0.7302],
         [0.4672, 0.2074, 0.8092,  ..., 0.6079, 0.3725, 0.9589]]),
 tensor([[0.6074, 0.9158, 0.9125,  ..., 0.2981, 0.4868, 0.5553],
         [0.0655, 0.7660, 0.8298,  ..., 0.5172, 0.4867, 0.6437],
         [0.4228, 0.8020, 0.0448,  ..., 0.0655, 0.0715, 0.

In [16]:
torch.save(random_acts,"random_acts_linear_correlated_model.pt")

In [18]:
acts_dict = torch.load("acts_linear_correlated_model_with_targets.pt")
acts_dict

<ipython-input-18-822fde6549f2>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  acts_dict = torch.load("acts_linear_correlated_model_with_targets.pt")


{'activations': [tensor([[ 0.1033,  0.0685, -0.4336,  ...,  0.1385, -0.1359,  0.3821],
          [ 0.1226,  0.1256,  0.7775,  ...,  0.0324, -0.1139,  0.3677],
          [ 0.1601,  0.0915, -0.2226,  ...,  0.0770, -0.1672,  0.3046],
          ...,
          [ 0.1105,  0.1304,  0.3942,  ...,  0.0648, -0.1310,  0.3294],
          [ 0.1157,  0.0306, -0.8115,  ...,  0.1040, -0.1949,  0.2510],
          [ 0.1229,  0.0810, -0.4630,  ...,  0.0964, -0.1829,  0.3757]]),
  tensor([[0.1033, 0.0685, 0.0000,  ..., 0.1385, 0.0000, 0.3821],
          [0.1226, 0.1256, 0.7775,  ..., 0.0324, 0.0000, 0.3677],
          [0.1601, 0.0915, 0.0000,  ..., 0.0770, 0.0000, 0.3046],
          ...,
          [0.1105, 0.1304, 0.3942,  ..., 0.0648, 0.0000, 0.3294],
          [0.1157, 0.0306, 0.0000,  ..., 0.1040, 0.0000, 0.2510],
          [0.1229, 0.0810, 0.0000,  ..., 0.0964, 0.0000, 0.3757]]),
  tensor([[ 0.3218,  0.9969,  0.0161,  ...,  0.2392,  0.3475, -0.5160],
          [ 0.6545,  0.6070, -0.0083,  ...,  0.8232

In [40]:
import torch
import random

def create_random_copy(original_dict):
  random_copy = {}
  for key, value in original_dict.items():
    if key == "activations":
      if isinstance(value, list):
        random_copy[key] = [torch.rand_like(tensor) for tensor in value]
    else:
      random_tensor = torch.rand_like(value.type(torch.float32))
      random_copy[key] = (random_tensor < 0.4).type(value.dtype)
  return random_copy

random_acts_dict = create_random_copy(acts_dict)
random_acts_dict

{'activations': [tensor([[0.0046, 0.8110, 0.2796,  ..., 0.6521, 0.1542, 0.8879],
          [0.1538, 0.9856, 0.1430,  ..., 0.6048, 0.7632, 0.4440],
          [0.6207, 0.9278, 0.2906,  ..., 0.4647, 0.6577, 0.3134],
          ...,
          [0.7374, 0.9312, 0.5501,  ..., 0.1062, 0.6272, 0.6866],
          [0.7961, 0.1136, 0.4682,  ..., 0.2393, 0.7829, 0.6014],
          [0.2546, 0.4837, 0.3173,  ..., 0.9429, 0.1216, 0.2678]]),
  tensor([[0.8445, 0.5189, 0.4191,  ..., 0.9432, 0.4562, 0.7760],
          [0.6090, 0.4492, 0.1626,  ..., 0.0535, 0.1005, 0.6667],
          [0.4657, 0.9736, 0.5929,  ..., 0.4094, 0.7204, 0.2619],
          ...,
          [0.1660, 0.6429, 0.5729,  ..., 0.6227, 0.9861, 0.1468],
          [0.2763, 0.8009, 0.5358,  ..., 0.4573, 0.9653, 0.1856],
          [0.6861, 0.0636, 0.1797,  ..., 0.1431, 0.6468, 0.7620]]),
  tensor([[0.3718, 0.1924, 0.9873,  ..., 0.2180, 0.0188, 0.1556],
          [0.8457, 0.8849, 0.4067,  ..., 0.5299, 0.0530, 0.0364],
          [0.5436, 0.5928, 

In [41]:
torch.save(random_acts_dict,"random_acts_linear_correlated_model_with_targets.pt")